# Machine Learning-Based Prediction of Tourist Arrivals in Sri Lanka Using Economic, Environmental and Tourism Factors

### ET-3043 Machine Learning Final Assignment

**Problem type:** Regression  
**Target variable:** `totalCount`  
**Models used:** Linear Regression and Polynomial Regression

## 1. Project Overview

This notebook predicts monthly tourist arrivals to Sri Lanka by origin country using economic, environmental, and tourism-related factors. The work follows the assignment requirements for a regression problem: data preparation, exploratory data analysis, model training, validation/testing, and evaluation using MAE, MSE, RMSE, and R2 Score.

## 2. Import Libraries

In [ ]:
# Data handling
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning preprocessing
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Regression models
from sklearn.linear_model import LinearRegression

# Evaluation metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid")

: 

## 3. Load and Understand the Dataset

In [ ]:
df = pd.read_csv("touristData.csv")

df.head()

In [ ]:
df.info()

In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

In [ ]:
df.describe()

In [ ]:
data_quality = pd.DataFrame({
    "Missing Values": df.isnull().sum(),
    "Data Type": df.dtypes
})

data_quality

In [ ]:
print("Duplicate rows:", df.duplicated().sum())

## 4. Exploratory Data Analysis

These plots summarize the target distribution, time trend, country contribution, monthly seasonality, and numeric feature relationships.

In [ ]:
month_order = [
    'January', 'February', 'March',
    'April', 'May', 'June',
    'July', 'August', 'September',
    'October', 'November', 'December'
]

month_mapping = {
    month: index
    for index, month in enumerate(month_order, start=1)
}


df['date'] = pd.to_datetime(
    df['year'].astype(str) + '-' + df['month'] + '-01'
)

In [ ]:
plt.figure(figsize=(8, 5))

sns.histplot(
    df['totalCount'],
    bins=30,
    kde=True
)

plt.title("Distribution of Tourist Arrivals")
plt.xlabel("Tourist Arrivals")
plt.ylabel("Frequency")

plt.show()

In [ ]:
monthly_total = df.groupby('date')['totalCount'].sum()

plt.figure(figsize=(12, 5))
monthly_total.plot(marker='o')

plt.title("Monthly Tourist Arrivals in Sri Lanka")
plt.xlabel("Year")
plt.ylabel("Number of Tourists")
plt.grid(True)

plt.show()

In [ ]:
country_total = (
    df.groupby('originCountry')['totalCount']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

plt.figure(figsize=(10, 5))
country_total.plot(kind='bar')

plt.title("Top 10 Tourist Origin Countries")
plt.xlabel("Origin Country")
plt.ylabel("Total Arrivals")
plt.xticks(rotation=45, ha='right')

plt.show()

In [ ]:
monthly_pattern = (
    df.groupby('month')['totalCount']
    .mean()
    .reindex(month_order)
)

plt.figure(figsize=(10, 5))
monthly_pattern.plot(marker='o')

plt.title("Average Tourist Arrivals by Month")
plt.xlabel("Month")
plt.ylabel("Average Arrivals")
plt.xticks(rotation=45, ha='right')
plt.grid(True)

plt.show()

In [ ]:
plt.figure(figsize=(12, 8))

sns.heatmap(
    df.select_dtypes(include=np.number).corr(),
    annot=True,
    cmap='coolwarm'
)

plt.title("Feature Correlation Matrix")

plt.show()

## 5. Feature Preparation

The target variable is `totalCount`. The `date` column is used only for time-based splitting and visualization, so it is removed from model features. `month` and `originCountry` are categorical for Linear Regression. For Polynomial Regression, a second-degree polynomial term is applied only to `month` to avoid creating too many unstable polynomial interactions.

In [ ]:
corr_matrix = df.select_dtypes(include=np.number).corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [column for column in upper.columns if any(upper[column] > 0.95)]
df_reduced = df.drop(to_drop, axis=1)

In [ ]:
X = df.drop(
    columns=['totalCount', 'date']
)

y = df['totalCount']

# Convert month names to ordered month numbers for model input.
X['month'] = X['month'].map(month_mapping)

In [ ]:
categorical_features = [
    'month',
    'originCountry'
]

numerical_features = [
    col for col in X.columns
    if col not in categorical_features
]

print("Numerical features:", numerical_features)
print("Categorical features:", categorical_features)

In [ ]:
linear_preprocessor = ColumnTransformer(
    transformers=[
        (
            'num',
            StandardScaler(),
            numerical_features
        ),
        (
            'cat',
            OneHotEncoder(
                handle_unknown='ignore',
                sparse_output=False
            ),
            categorical_features
        )
    ]
)


polynomial_preprocessor = ColumnTransformer(
    transformers=[
        (
            'num',
            StandardScaler(),
            [
                col for col in X.columns
                if col not in ['month', 'originCountry']
            ]
        ),
        (
            'month_poly',
            Pipeline(
                steps=[
                    ('scaler', StandardScaler()),
                    ('polynomial_features', PolynomialFeatures(
                        degree=2,
                        include_bias=False
                    ))
                ]
            ),
            ['month']
        ),
        (
            'cat',
            OneHotEncoder(
                handle_unknown='ignore',
                sparse_output=False
            ),
            ['originCountry']
        )
    ]
)

## 6. Training, Validation, and Testing Split

Because the dataset is time-based, a chronological split is used instead of random splitting. This prevents future information from leaking into model training.

In [ ]:
train = df[df.year <= 2022].copy()
validation = df[df.year == 2023].copy()
test = df[df.year == 2024].copy()

X_train = X.loc[train.index].copy()
y_train = train['totalCount']

X_val = X.loc[validation.index].copy()
y_val = validation['totalCount']

X_test = X.loc[test.index].copy()
y_test = test['totalCount']

split_summary = pd.DataFrame({
    "Split": ["Train", "Validation", "Test"],
    "Years": [
        f"{train.year.min()}-{train.year.max()}",
        str(validation.year.unique()[0]),
        str(test.year.unique()[0])
    ],
    "Rows": [len(train), len(validation), len(test)]
})

split_summary

## 7. Model Training

Only the lecture-approved regression models are used: Linear Regression and Polynomial Regression.

In [ ]:
linear_model = Pipeline(
    steps=[
        ('preprocessor', linear_preprocessor),
        ('model', LinearRegression())
    ]
)


polynomial_model = Pipeline(
    steps=[
        ('preprocessor', polynomial_preprocessor),
        ('model', LinearRegression())
    ]
)


models = {
    "Linear Regression": linear_model,
    "Polynomial Regression": polynomial_model
}

## 8. Performance Evaluation

For this regression problem, the assignment requires MAE, MSE, RMSE, and R2 Score.

In [ ]:
def evaluate_model(model, X_eval, y_eval):
    predictions = model.predict(X_eval)

    mae = mean_absolute_error(y_eval, predictions)
    mse = mean_squared_error(y_eval, predictions)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_eval, predictions)

    return {
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2 Score": r2
    }

In [ ]:
rows = []

for name, model in models.items():
    model.fit(X_train, y_train)

    for split_name, X_eval, y_eval in [
        ('Validation 2023', X_val, y_val),
        ('Test 2024', X_test, y_test)
    ]:
        score = evaluate_model(model, X_eval, y_eval)
        score['Model'] = name
        score['Split'] = split_name
        rows.append(score)


results = (
    pd.DataFrame(rows)
    .set_index(['Model', 'Split'])
    .sort_values('RMSE')
)

results

## 9. Model Comparison and Error Analysis

The evaluation table above is useful, but a stronger report should also compare feature choices and inspect the prediction errors directly.


In [ ]:
reduced_feature_subset = [
    'month',
    'originCountry',
    'apparent_temperature_mean_celcius',
    'sunshine_duration_seconds',
    'rain_sum_mm',
    'precipitation_hours',
    'num_establishments',
    'consumerPriceIndex'
]

X_reduced = X[reduced_feature_subset].copy()

feature_comparison_rows = []

for feature_label, X_subset in [
    ('Full features', X),
    ('Reduced features', X_reduced)
]:
    categorical_features = ['month', 'originCountry']
    numerical_features = [col for col in X_subset.columns if col not in categorical_features]

    feature_preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), numerical_features),
            ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
        ]
    )

    feature_model = Pipeline(
        steps=[
            ('preprocessor', feature_preprocessor),
            ('model', LinearRegression())
        ]
    )

    feature_model.fit(X_train, y_train)

    for split_name, X_eval, y_eval in [
        ('Validation 2023', X_val, y_val),
        ('Test 2024', X_test, y_test)
    ]:
        score = evaluate_model(feature_model, X_eval, y_eval)
        score['Feature Set'] = feature_label
        score['Split'] = split_name
        feature_comparison_rows.append(score)

feature_comparison_results = (
    pd.DataFrame(feature_comparison_rows)
    .set_index(['Feature Set', 'Split'])
    .sort_values(['Split', 'RMSE'])
)

feature_comparison_results


## 9. Actual vs Predicted Visualization

In [ ]:
best_model_name = results.xs('Test 2024', level='Split')['RMSE'].idxmin()
best_model = models[best_model_name]
best_predictions = best_model.predict(X_test)

residuals = pd.DataFrame({
    'Actual': y_test,
    'Predicted': best_predictions,
    'Residual': y_test - best_predictions
})

plt.figure(figsize=(8, 5))
plt.scatter(
    residuals['Predicted'],
    residuals['Residual'],
    alpha=0.75
)
plt.axhline(0, color='red', linestyle='--', label='Zero error')
plt.xlabel('Predicted Tourist Arrivals')
plt.ylabel('Residual')
plt.title(f'Residual Plot for {best_model_name}')
plt.legend()
plt.grid(True)
plt.show()

residuals.head()


## 10. Results Analysis Notes

Use the evaluation table and scatter plot above in the IEEE paper discussion. In the current experiment, Linear Regression performs better than Polynomial Regression on the 2024 test set. The model captures part of the arrival pattern, but prediction error remains noticeable because tourist arrivals changed strongly during and after the COVID-19 period, and the dataset is relatively small.

## 11. Execution Instructions

Run the notebook from top to bottom in a Python environment with pandas, numpy, matplotlib, seaborn, and scikit-learn installed. The file `touristData.csv` must be in the same folder as this notebook.